# 🏆 2026 FIFA World Cup Prediction Model
**Deterministic Scoring (2022) → Machine Learning (2026)**

*Carlota Huertas · 2025*

---

An evolution of the [2022 World Cup prediction project](https://github.com/carlotahuertas/PredictingWorldCup) that used a hand-crafted 70/30 weighted player-rating algorithm to deterministically simulate the bracket.

This notebook keeps that exact player-scoring foundation and upgrades the prediction engine: two competing ML models — **Logistic Regression** and **Random Forest** — are trained on 900+ historical World Cup matches to produce **probability distributions** per match rather than a single fixed outcome.

| | 2022 | 2026 |
|---|---|---|
| Teams | 32 | **48** (expanded format) |
| Prediction method | Rule-based score comparison | Logistic Regression + Random Forest |
| Output | Definite bracket | Win probability per match |
| Training data | None (no learning) | 900+ historical WC matches |
| Uncertainty | None | Confidence scores modelled |
| Format handled | 8 groups of 4 | 12 groups of 4 + 8 best 3rd-place teams |

In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import requests
import json
import os
import re
import warnings
from pathlib import Path
from datetime import datetime
from itertools import combinations
from collections import defaultdict

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import accuracy_score

try:
    from bs4 import BeautifulSoup
    BS4_AVAILABLE = True
except ImportError:
    BS4_AVAILABLE = False
    print("⚠️  bs4 not found — pip install beautifulsoup4")

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

try:
    import kaleido
    KALEIDO_AVAILABLE = True
except ImportError:
    KALEIDO_AVAILABLE = False
    print("⚠️  kaleido not found — Plotly PNG export disabled. pip install kaleido")

warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ Imports complete")

✅ Imports complete


In [12]:
DATA_DIR = Path('data')
VIZ_DIR  = Path('visualizations')
DATA_DIR.mkdir(exist_ok=True)
VIZ_DIR.mkdir(exist_ok=True)

API_KEY  = os.getenv('FOOTBALL_DATA_API_KEY', '')
API_BASE = 'https://api.football-data.org/v4'

# Portfolio colour palette
C = {
    'accent': '#EF6545',  # Canned Tomato
    'cool':   '#AECFD0',  # Dip at Twilight
    'warm':   '#F49625',  # Iced Tang
    'dark':   '#422F0E',  # Valentine Chocolate
    'bg':     '#F5E6C0',  # Buttered Corn
    'teal':   '#037F71',  # Bali Pool
    'muted':  '#7A5C50',
}
CONF_COLORS = {
    'UEFA': C['cool'], 'CONMEBOL': C['accent'], 'AFC': C['warm'],
    'CAF':  C['teal'], 'CONCACAF': C['dark'],  'OFC': C['muted'],
}

N_GROUPS        = 12
TEAMS_PER_GROUP = 4
TOP_THIRDS      = 8   # best third-place teams advancing to R32

print(f"🔑 API key: {'✓ found' if API_KEY else '✗ not set — fallback data will be used'}")
print(f"📁 Data dir: {DATA_DIR.resolve()}")

🔑 API key: ✗ not set — fallback data will be used
📁 Data dir: /Users/carlotahuertas/Desktop/carlota-huertas/data


---
## Part 0: Data Cleaning

Before any modeling, the raw FIFA dataset needs to be understood, cleaned,
and shaped into something meaningful. This section documents every decision made.

**Source**: EA Sports FC (FIFA 15–24) complete player dataset — 180,021 rows across
10 game editions, 109 columns.

**Key findings from initial inspection:**
- The CSV contains FIFA 15 through FIFA 24 stacked in one file (`fifa_version` column)
- Only FIFA 24 (18,350 players) is relevant — it's the most recent edition
- 19 of 48 World Cup teams have confirmed national squad data via `nation_position` column
- 29 teams have no `nation_position` data — we use top-26-by-overall as a proxy
- 4 team name mismatches between our group draw and the CSV require normalization
- `nation_position` contains SUB/RES markers alongside actual positions — these need mapping

In [13]:
# ── STEP 0.1 — Load and filter to FIFA 24 only ───────────────────────────────
print("=" * 60)
print("STEP 0.1 — Loading raw data")
print("=" * 60)

RAW_PATH = DATA_DIR / 'fifa25_players.csv'

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"FIFA dataset not found at {RAW_PATH}\n"
        "Download from: https://www.kaggle.com/datasets/stefanoleone992/"
        "ea-sports-fc-25-complete-player-dataset\n"
        "Save as data/fifa25_players.csv"
    )

df_raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Raw dataset: {len(df_raw):,} rows x {len(df_raw.columns)} columns")
print(f"FIFA versions present: {sorted(df_raw['fifa_version'].dropna().unique())}")

df24 = df_raw[df_raw['fifa_version'] == 24].copy()
print(f"\nAfter filtering to FIFA 24: {len(df24):,} players")
print(f"Nations represented: {df24['nationality_name'].nunique()}")

STEP 0.1 — Loading raw data


FileNotFoundError: FIFA dataset not found at data/fifa25_players.csv
Download from: https://www.kaggle.com/datasets/stefanoleone992/ea-sports-fc-25-complete-player-dataset
Save as data/fifa25_players.csv

In [ ]:
# ── STEP 0.2 — Team name normalization ───────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 0.2 — Team name normalization")
print("=" * 60)

NAME_MAP = {
    'Korea Republic':     'South Korea',
    "Côte d'Ivoire":      'Ivory Coast',
    "Cote d'Ivoire":      'Ivory Coast',
    'Cape Verde Islands': 'Cape Verde',
    'Congo DR':           'DR Congo',
}

df24['nationality_name'] = df24['nationality_name'].replace(NAME_MAP)

print(f"Name mappings applied: {len(NAME_MAP)}")
for old, new in NAME_MAP.items():
    count = len(df_raw[df_raw['nationality_name'] == old])
    print(f"  '{old}' -> '{new}' ({count} players affected)")

In [ ]:
# ── STEP 0.3 — Define 48 WC teams and check coverage ─────────────────────────
print("\n" + "=" * 60)
print("STEP 0.3 — World Cup 2026 team coverage check")
print("=" * 60)

WC2026_TEAMS = [
    # Group A
    'Mexico', 'South Africa', 'South Korea', 'Czech Republic',
    # Group B
    'Canada', 'Switzerland', 'Qatar', 'Bosnia and Herzegovina',
    # Group C
    'Brazil', 'Morocco', 'Scotland', 'Haiti',
    # Group D
    'United States', 'Paraguay', 'Australia', 'Turkey',
    # Group E
    'Germany', 'Curacao', 'Ivory Coast', 'Ecuador',
    # Group F
    'Netherlands', 'Japan', 'Tunisia', 'Sweden',
    # Group G
    'Belgium', 'Egypt', 'Iran', 'New Zealand',
    # Group H
    'Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay',
    # Group I
    'France', 'Senegal', 'Norway', 'Iraq',
    # Group J
    'Argentina', 'Algeria', 'Austria', 'Jordan',
    # Group K
    'Portugal', 'Colombia', 'Uzbekistan', 'DR Congo',
    # Group L
    'England', 'Croatia', 'Ghana', 'Panama',
]

print(f"Teams in tournament: {len(WC2026_TEAMS)}")
print()

missing_from_csv = []
for team in WC2026_TEAMS:
    count = len(df24[df24['nationality_name'] == team])
    if count == 0:
        missing_from_csv.append(team)
        print(f"  x {team}: NOT FOUND in CSV")
    else:
        print(f"  v {team}: {count} players in FIFA 24")

if missing_from_csv:
    print(f"\n  {len(missing_from_csv)} teams not found -- will use FIFA ranking fallback scores")

In [ ]:
# ── STEP 0.4 — Position mapping (nation_position -> GK/DF/MF/FW) ─────────────
print("\n" + "=" * 60)
print("STEP 0.4 — Position normalization")
print("=" * 60)

POSITION_MAP = {
    'GK': 'GK',
    'CB': 'DF', 'LCB': 'DF', 'RCB': 'DF',
    'LB': 'DF', 'RB': 'DF', 'LWB': 'DF', 'RWB': 'DF',
    'CM': 'MF', 'LCM': 'MF', 'RCM': 'MF',
    'CDM': 'MF', 'LDM': 'MF', 'RDM': 'MF',
    'CAM': 'MF', 'LAM': 'MF', 'RAM': 'MF',
    'LM': 'MF', 'RM': 'MF',
    'ST': 'FW', 'LS': 'FW', 'RS': 'FW',
    'LW': 'FW', 'RW': 'FW', 'LF': 'FW', 'RF': 'FW', 'CF': 'FW',
}

def infer_pos(pos_str):
    if pd.isna(pos_str):
        return 'MF'
    return POSITION_MAP.get(str(pos_str).split(',')[0].strip(), 'MF')

def map_nation_position(row):
    nat_pos = row.get('nation_position')
    if pd.isna(nat_pos) or str(nat_pos) in ('SUB', 'RES'):
        return infer_pos(row.get('player_positions'))
    return POSITION_MAP.get(str(nat_pos).strip(), infer_pos(row.get('player_positions')))

print("Position mapping defined:")
print("  Goalkeepers:  GK")
print("  Defenders:    CB, LCB, RCB, LB, RB, LWB, RWB")
print("  Midfielders:  CM, CDM, CAM, LM, RM + L/R variants")
print("  Forwards:     ST, LW, RW, CF, LF, RF + variants")
print("  SUB / RES:    inferred from player_positions (first listed)")

In [ ]:
# ── STEP 0.5 — Build clean squad dataset (26 players per team) ───────────────
print("\n" + "=" * 60)
print("STEP 0.5 — Building clean squad dataset")
print("=" * 60)

TEAMS_WITH_SQUADS = []
squads = []

for team in WC2026_TEAMS:
    tdf = df24[df24['nationality_name'] == team].copy()

    if len(tdf) == 0:
        print(f"  SKIP {team} -- no data in CSV")
        continue

    tdf_squad = tdf[tdf['nation_position'].notna()].copy()

    if len(tdf_squad) >= 20:
        strategy = 'nation_position (confirmed squad)'
        TEAMS_WITH_SQUADS.append(team)
    else:
        tdf_squad = tdf.nlargest(26, 'overall').copy()
        strategy = 'top-26 by overall (estimated squad)'

    tdf_squad['pos_cat']         = tdf_squad.apply(map_nation_position, axis=1)
    tdf_squad['squad_strategy']  = strategy
    squads.append(tdf_squad)
    print(f"  v {team}: {len(tdf_squad)} players [{strategy}]")

df_squads = pd.concat(squads, ignore_index=True)
print(f"\nTotal squad players: {len(df_squads)}")
print(f"Teams with confirmed squads: {len(TEAMS_WITH_SQUADS)}")
print(f"Teams using top-26 estimate: {len(WC2026_TEAMS) - len(TEAMS_WITH_SQUADS) - len(missing_from_csv)}")

In [ ]:
# ── STEP 0.6 — Validate squad composition ────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 0.6 — Squad composition validation")
print("=" * 60)

issues = []
for team in WC2026_TEAMS:
    tdf = df_squads[df_squads['nationality_name'] == team]
    if len(tdf) == 0:
        continue
    pos_counts = tdf['pos_cat'].value_counts().to_dict()
    gk = pos_counts.get('GK', 0)
    df_ = pos_counts.get('DF', 0)
    mf = pos_counts.get('MF', 0)
    fw = pos_counts.get('FW', 0)
    flag = ''
    if gk == 0:
        flag = '  NO GOALKEEPER'
        issues.append(team)
    elif gk > 5:
        flag = f'  Too many GKs ({gk})'
        issues.append(team)
    print(f"  {team}: GK={gk} DF={df_} MF={mf} FW={fw}{flag}")

if issues:
    print(f"\n  Teams with composition issues: {issues}")
else:
    print("\n  All squad compositions look reasonable")

In [ ]:
# ── STEP 0.7 — Handle missing values in key rating columns ───────────────────
print("\n" + "=" * 60)
print("STEP 0.7 — Missing value audit")
print("=" * 60)

KEY_COLS = [
    'overall', 'pace', 'physic', 'shooting', 'passing', 'defending',
    'goalkeeping_reflexes', 'goalkeeping_positioning', 'goalkeeping_handling',
    'defending_marking_awareness', 'movement_agility', 'skill_long_passing',
    'mentality_interceptions', 'attacking_heading_accuracy', 'power_strength',
]

print("Missing values per key column:")
for col in KEY_COLS:
    missing = df_squads[col].isna().sum() if col in df_squads.columns else len(df_squads)
    pct = missing / len(df_squads) * 100
    print(f"  {col}: {missing} missing ({pct:.1f}%)")

print("\nFilling missing values with position-level medians:")
for col in KEY_COLS:
    if col not in df_squads.columns:
        df_squads[col] = 50.0
        print(f"  {col}: column missing -- filled with 50")
        continue
    before = df_squads[col].isna().sum()
    if before > 0:
        df_squads[col] = df_squads[col].fillna(
            df_squads.groupby('pos_cat')[col].transform('median')
        )
        df_squads[col] = df_squads[col].fillna(df_squads[col].median())
        print(f"  {col}: filled {before} nulls")

remaining = df_squads[KEY_COLS].isna().sum().sum()
print(f"\nRemaining nulls across all key columns: {remaining}")

In [ ]:
# ── STEP 0.8 — Type validation and range checks ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 0.8 — Type validation and range checks")
print("=" * 60)

KEY_COLS = [
    'overall', 'pace', 'physic', 'shooting', 'passing', 'defending',
    'goalkeeping_reflexes', 'goalkeeping_positioning', 'goalkeeping_handling',
    'defending_marking_awareness', 'movement_agility', 'skill_long_passing',
    'mentality_interceptions', 'attacking_heading_accuracy', 'power_strength',
]

for col in KEY_COLS:
    df_squads[col] = pd.to_numeric(df_squads[col], errors='coerce').fillna(50.0)

print("Rating column ranges (expected: 1-99):")
for col in KEY_COLS:
    mn = df_squads[col].min(); mx = df_squads[col].max()
    oob = ((df_squads[col] < 1) | (df_squads[col] > 99)).sum()
    flag = '  !' if oob > 0 else '  v'
    print(f"{flag} {col}: min={mn:.0f}, max={mx:.0f}, out-of-range={oob}")

In [ ]:
# ── STEP 0.9 — Save clean dataset and print summary ──────────────────────────
print("\n" + "=" * 60)
print("STEP 0.9 — Save clean dataset")
print("=" * 60)

KEY_COLS = [
    'overall', 'pace', 'physic', 'shooting', 'passing', 'defending',
    'goalkeeping_reflexes', 'goalkeeping_positioning', 'goalkeeping_handling',
    'defending_marking_awareness', 'movement_agility', 'skill_long_passing',
    'mentality_interceptions', 'attacking_heading_accuracy', 'power_strength',
]

KEEP_COLS = [
    'short_name', 'long_name', 'nationality_name', 'overall',
    'pos_cat', 'squad_strategy', 'nation_position', 'player_positions',
] + KEY_COLS

available_cols = [c for c in KEEP_COLS if c in df_squads.columns]
df_clean = df_squads[available_cols].copy()

CLEAN_PATH = DATA_DIR / 'wc2026_squads.csv'
df_clean.to_csv(CLEAN_PATH, index=False)

print(f"Saved: {CLEAN_PATH}")
print(f"  Rows: {len(df_clean)}")
print(f"  Columns: {len(df_clean.columns)}")
print()
print("=" * 60)
print("DATA CLEANING COMPLETE")
print("=" * 60)
print(f"  Input:  {len(df_raw):,} rows (FIFA 15-24, all nations)")
print(f"  Output: {len(df_clean):,} rows (FIFA 24, WC nations, clean squads)")
print(f"  Reduction: {(1 - len(df_clean)/len(df_raw))*100:.1f}% of rows removed")
print()
print(f"  Teams with confirmed squads (nation_position): {len(TEAMS_WITH_SQUADS)}")
print(f"  Teams using top-26 estimate: {len([t for t in WC2026_TEAMS if t not in TEAMS_WITH_SQUADS and t not in missing_from_csv])}")
print(f"  Teams using ranking fallback (no CSV data):   {len(missing_from_csv)}")
print()
print("  Decisions made:")
print("  1. Filtered to FIFA 24 only (most recent ratings)")
print("  2. Normalised 4 team name mismatches")
print("  3. Used nation_position for teams with confirmed squad data")
print("  4. Used top-26-by-overall for remaining teams (proxy for squad)")
print("  5. Mapped all position codes -> GK/DF/MF/FW")
print("  6. Filled nulls with position-level medians (not global medians)")
print("  7. Validated all rating columns are numeric, range 1-99")

# Expose for downstream use
df_players = df_clean

---
## Part 1: Data Collection

Three data sources power this notebook:

1. **Historical WC matches** — [`martj42/international_results`](https://github.com/martj42/international_results) public CSV (~900 World Cup matches, 1930–2022). This is the **training corpus** for both ML models.
2. **2026 group draw & teams** — `football-data.org` API (primary) → Wikipedia scrape (fallback) → hardcoded estimate (last resort).
3. **Player ratings** — FIFA 25 dataset from Kaggle (preferred) → FIFA ranking-based strength scores (fallback).

The player-scoring method (70/30 weighted algorithm) is **identical to the 2022 project** — this is deliberate. It isolates the prediction-engine upgrade from the team-strength inputs.

In [ ]:
HIST_URL  = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
HIST_PATH = DATA_DIR / "historical_results.csv"

if not HIST_PATH.exists():
    print("⬇️  Downloading historical match data...")
    df_hist = pd.read_csv(HIST_URL)
    df_hist.to_csv(HIST_PATH, index=False)
    print(f"   Saved {len(df_hist):,} rows → {HIST_PATH}")
else:
    df_hist = pd.read_csv(HIST_PATH)
    print(f"📂 Loaded from cache: {len(df_hist):,} matches")

# Filter to FIFA World Cup matches only
df_wc = df_hist[df_hist['tournament'] == 'FIFA World Cup'].copy()
df_wc['date'] = pd.to_datetime(df_wc['date'])
df_wc = df_wc.sort_values('date').reset_index(drop=True)

# Encode result from home-team perspective: 0=home win, 1=draw, 2=away win
df_wc['result'] = df_wc.apply(
    lambda r: 0 if r['home_score'] > r['away_score']
              else (1 if r['home_score'] == r['away_score'] else 2),
    axis=1
)

print(f"\n⚽ World Cup matches: {len(df_wc)}")
print(f"   Years: {df_wc['date'].dt.year.min()}–{df_wc['date'].dt.year.max()}")
print(f"   Result split: {dict(df_wc['result'].value_counts().sort_index().rename({0:'Home W',1:'Draw',2:'Away W'}))}")
df_wc[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'result']].tail(6)

In [ ]:
# ── Confederation mapping for all 48 teams ────────────────────────────────────
TEAM_CONF = {
    # CONMEBOL
    'Argentina':'CONMEBOL','Brazil':'CONMEBOL','Colombia':'CONMEBOL',
    'Uruguay':'CONMEBOL','Ecuador':'CONMEBOL','Paraguay':'CONMEBOL',
    # CONCACAF
    'United States':'CONCACAF','Mexico':'CONCACAF','Canada':'CONCACAF',
    'Panama':'CONCACAF','Haiti':'CONCACAF','Curacao':'CONCACAF',
    # UEFA
    'France':'UEFA','England':'UEFA','Portugal':'UEFA','Spain':'UEFA',
    'Netherlands':'UEFA','Germany':'UEFA','Croatia':'UEFA','Denmark':'UEFA',
    'Switzerland':'UEFA','Austria':'UEFA','Turkey':'UEFA',
    'Czech Republic':'UEFA','Belgium':'UEFA','Scotland':'UEFA',
    'Norway':'UEFA','Sweden':'UEFA','Bosnia and Herzegovina':'UEFA',
    # AFC
    'Japan':'AFC','South Korea':'AFC','Iran':'AFC','Australia':'AFC',
    'Saudi Arabia':'AFC','Iraq':'AFC','Jordan':'AFC','Uzbekistan':'AFC',
    'Qatar':'AFC',
    # CAF
    'Morocco':'CAF','Senegal':'CAF','Egypt':'CAF',
    'Ivory Coast':'CAF','Ghana':'CAF','South Africa':'CAF',
    'Tunisia':'CAF','Algeria':'CAF','Cape Verde':'CAF','DR Congo':'CAF',
    # OFC
    'New Zealand':'OFC',
}

# ── Source 1: football-data.org API ───────────────────────────────────────────
def fetch_groups_api():
    if not API_KEY:
        return None
    try:
        r = requests.get(f"{API_BASE}/competitions/WC2026/groups",
                         headers={'X-Auth-Token': API_KEY}, timeout=10)
        if r.status_code == 200:
            data = r.json()
            groups = {}
            for g in data.get('groups', []):
                name = g.get('name', g.get('id', 'Group?'))
                teams = [t['name'] for t in g.get('teams', [])]
                if teams:
                    groups[name] = teams
            if groups:
                print(f"✅ API: {len(groups)} groups loaded")
                return groups
    except Exception as e:
        print(f"   API error: {e}")
    return None

# ── Source 2: Wikipedia scrape ─────────────────────────────────────────────────
def fetch_groups_wikipedia():
    if not BS4_AVAILABLE:
        return None
    url = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_group_stage"
    try:
        r = requests.get(url, timeout=15, headers={'User-Agent': 'Mozilla/5.0'})
        soup = BeautifulSoup(r.content, 'html.parser')
        groups = {}
        for tag in soup.find_all(['h2', 'h3']):
            txt = tag.get_text(strip=True)
            m = re.match(r'Group ([A-L])', txt)
            if m:
                letter = m.group(1)
                tbl = tag.find_next('table', class_='wikitable')
                if tbl:
                    teams = []
                    for row in tbl.find_all('tr')[1:]:
                        cells = row.find_all(['td', 'th'])
                        if cells:
                            name = re.sub(r'\[.*?\]', '', cells[0].get_text(strip=True))
                            if name and len(name) > 1:
                                teams.append(name)
                    if teams:
                        groups[f'Group {letter}'] = teams[:4]
        if len(groups) >= 8:
            print(f"✅ Wikipedia: {len(groups)} groups scraped")
            return groups
    except Exception as e:
        print(f"   Wikipedia error: {e}")
    return None

# ── Source 3: Official 2026 draw (verified) ────────────────────────────────────
# From the December 5, 2024 draw. Matches WC2026_TEAMS defined in Part 0.
FALLBACK_GROUPS = {
    'Group A': ['Mexico',        'South Africa', 'South Korea',           'Czech Republic'],
    'Group B': ['Canada',        'Switzerland',  'Qatar',                 'Bosnia and Herzegovina'],
    'Group C': ['Brazil',        'Morocco',      'Scotland',              'Haiti'],
    'Group D': ['United States', 'Paraguay',     'Australia',             'Turkey'],
    'Group E': ['Germany',       'Curacao',      'Ivory Coast',           'Ecuador'],
    'Group F': ['Netherlands',   'Japan',        'Tunisia',               'Sweden'],
    'Group G': ['Belgium',       'Egypt',        'Iran',                  'New Zealand'],
    'Group H': ['Spain',         'Cape Verde',   'Saudi Arabia',          'Uruguay'],
    'Group I': ['France',        'Senegal',      'Norway',                'Iraq'],
    'Group J': ['Argentina',     'Algeria',      'Austria',               'Jordan'],
    'Group K': ['Portugal',      'Colombia',     'Uzbekistan',            'DR Congo'],
    'Group L': ['England',       'Croatia',      'Ghana',                 'Panama'],
}

groups_2026 = fetch_groups_api() or fetch_groups_wikipedia() or FALLBACK_GROUPS
source_used = 'API' if fetch_groups_api.__doc__ else 'Wikipedia/Fallback'

all_teams_2026 = [t for teams in groups_2026.values() for t in teams]
print(f"\n📋 Total teams: {len(all_teams_2026)}")
print("Groups:")
for g, teams in groups_2026.items():
    print(f"  {g}: {', '.join(teams)}")

In [ ]:
# Data cleaning was handled in Part 0 — load the clean squad dataset.
# If Part 0 has not been run yet, do so before continuing.

CLEAN_PATH = DATA_DIR / 'wc2026_squads.csv'

# ── FIFA ranking-based strength scores (fallback for teams absent from CSV) ───
RANKING_SCORES = {
    'Argentina':57.0,'France':56.0,'England':55.0,'Brazil':54.5,
    'Portugal':54.0,'Spain':53.5,'Netherlands':53.0,'Belgium':52.0,
    'Germany':51.5,'Croatia':50.0,'Uruguay':49.5,'Colombia':49.0,
    'Morocco':48.0,'Switzerland':47.5,'Norway':48.5,'Japan':47.0,
    'Senegal':46.5,'Mexico':46.0,'United States':45.5,'South Korea':45.0,
    'Turkey':44.5,'Ecuador':44.0,'Australia':43.0,'Sweden':47.0,
    'Iran':42.5,'Canada':42.0,'Scotland':40.0,'Austria':40.0,
    'Ivory Coast':39.5,'Ghana':38.5,'Czech Republic':37.5,'Tunisia':36.5,
    'Egypt':36.0,'Saudi Arabia':35.5,'South Africa':35.0,'Algeria':35.0,
    'Paraguay':38.0,'Panama':33.5,'Qatar':34.0,'Bosnia and Herzegovina':33.0,
    'New Zealand':31.5,'Iraq':30.5,'Uzbekistan':30.0,'DR Congo':30.0,
    'Jordan':28.5,'Cape Verde':29.0,'Haiti':27.0,'Curacao':26.0,
}

if CLEAN_PATH.exists():
    df_players = pd.read_csv(CLEAN_PATH)
    RATINGS_SOURCE = 'FIFA 24 Kaggle dataset (cleaned in Part 0)'
    print(f"Loaded clean squads: {len(df_players):,} players across "
          f"{df_players['nationality_name'].nunique()} teams")
    print(f"Columns: {list(df_players.columns)}")
else:
    df_players = None
    RATINGS_SOURCE = 'FIFA ranking fallback scores'
    print("wc2026_squads.csv not found -- run Part 0 first.")
    print("Falling back to FIFA ranking-based scores.")

---
## Part 2: Data Preprocessing

Three preprocessing steps:

1. **Team strength scores** — the same 70/30 weighted algorithm as the 2022 project: 30% general attributes (overall, pace, physic) + 70% position-specific attributes. Produces one composite score per team.
2. **Elo ratings** — calculated rolling Elo from all 900+ historical WC matches (K=40). Gives a form-aware strength measure beyond static player ratings.
3. **Feature engineering** — builds the feature matrix used to train the ML models: `strength_diff`, `elo_diff`, `h2h_win_rate`, `h2h_draw_rate`.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
# SAME ALGORITHM AS 2022 PROJECT — kept identical intentionally.
# This isolates the improvement: same team-strength inputs,
# fundamentally more sophisticated prediction engine.
# ═════════════════════════════════════════════════════════════════════════

POSITION_WEIGHTS = {
    'GK': ['goalkeeping_reflexes', 'goalkeeping_positioning', 'goalkeeping_handling'],
    'DF': ['defending', 'defending_marking_awareness', 'movement_agility'],
    'MF': ['passing', 'skill_long_passing', 'mentality_interceptions'],
    'FW': ['attacking_heading_accuracy', 'shooting', 'power_strength'],
}
GENERAL_ATTRS   = ['overall', 'physic', 'pace']
GENERAL_WEIGHT  = 0.30
POSITION_WEIGHT = 0.70

def calculate_player_score(player_row, position):
    general  = player_row[GENERAL_ATTRS].astype(float).mean() * GENERAL_WEIGHT
    specific = player_row[POSITION_WEIGHTS[position]].astype(float).mean() * POSITION_WEIGHT
    return round(general + specific, 2)

if df_players is not None:
    df_players['player_score'] = df_players.apply(
        lambda r: calculate_player_score(r, r['pos_cat']), axis=1
    )
    print("Player scores calculated (70/30 algorithm)")
    print(df_players.groupby('nationality_name')['player_score'].mean().sort_values(ascending=False).head(10))

In [ ]:
def calculate_team_score(team_players_df):
    return round(team_players_df['player_score'].mean(), 2)

# Use WC2026_TEAMS if defined by Part 0, else fall back to all_teams_2026
_teams_ref = WC2026_TEAMS if 'WC2026_TEAMS' in dir() else all_teams_2026

team_scores      = {}
team_score_detail = []

for team in _teams_ref:
    if df_players is not None:
        # df_players already has pos_cat and player_score from Part 0 + Part 2 cell 9
        tdf = df_players[df_players['nationality_name'] == team]
    else:
        tdf = pd.DataFrame()

    if len(tdf) == 0 or 'player_score' not in tdf.columns:
        score = RANKING_SCORES.get(team, 35.0)
        team_scores[team] = score
        team_score_detail.append({'team': team, 'score': score, 'players': 0, 'source': 'ranking'})
        continue

    score = calculate_team_score(tdf)
    team_scores[team] = score
    pos_breakdown = tdf.groupby('pos_cat')['player_score'].mean().round(2).to_dict()
    team_score_detail.append({
        'team': team, 'score': score, 'players': len(tdf),
        'source': 'fifa24_cleaned', **pos_breakdown
    })

df_team_scores = pd.DataFrame(team_score_detail).sort_values('score', ascending=False).reset_index(drop=True)
df_team_scores.to_csv(DATA_DIR / 'team_scores.csv', index=False)

print(f"Team strength scores (source: {RATINGS_SOURCE})")
print(df_team_scores[['team', 'score', 'players', 'source']].to_string(index=False))

In [ ]:
def expected_elo(ra, rb):
    return 1.0 / (1.0 + 10.0 ** ((rb - ra) / 400.0))

def update_elo(rating, expected, actual, k=40):
    return rating + k * (actual - expected)

def calculate_elo_ratings(df):
    elos   = defaultdict(lambda: 1500.0)
    history = []
    for _, row in df.iterrows():
        h, a = row['home_team'], row['away_team']
        exp_h = expected_elo(elos[h], elos[a])
        exp_a = 1.0 - exp_h
        actual_h = 1.0 if row['result'] == 0 else (0.5 if row['result'] == 1 else 0.0)
        actual_a = 1.0 - actual_h
        elos[h] = update_elo(elos[h], exp_h, actual_h)
        elos[a] = update_elo(elos[a], exp_a, actual_a)
        history.append({'date': row['date'], 'home': h, 'away': a,
                        'elo_h': round(elos[h], 1), 'elo_a': round(elos[a], 1)})
    return dict(elos), pd.DataFrame(history)

team_elos, elo_df = calculate_elo_ratings(df_wc)

print(f"Elo ratings calculated from {len(df_wc)} historical WC matches")
print("\nTop 15 teams by Elo:")
top_elo = pd.Series(team_elos).sort_values(ascending=False).head(15)
for team, elo in top_elo.items():
    bar = '█' * int((elo - 1300) / 30)
    print(f"  {team:<22} {elo:>7.1f}  {bar}")

In [ ]:
def h2h_stats(df, team1, team2, n=5):
    mask = (
        ((df['home_team'] == team1) & (df['away_team'] == team2)) |
        ((df['home_team'] == team2) & (df['away_team'] == team1))
    )
    recent = df[mask].tail(n)
    if len(recent) == 0:
        return 0.0, 0.0
    wins = sum(
        (r['home_team'] == team1 and r['result'] == 0) or
        (r['away_team'] == team1 and r['result'] == 2)
        for _, r in recent.iterrows()
    )
    draws = sum(r['result'] == 1 for _, r in recent.iterrows())
    return wins / n, draws / n

# Build feature matrix from historical WC matches
elos_snapshot = defaultdict(lambda: 1500.0)
feature_rows = []

for _, row in df_wc.iterrows():
    h, a = row['home_team'], row['away_team']
    h_score = team_scores.get(h, np.mean(list(team_scores.values())))
    a_score = team_scores.get(a, np.mean(list(team_scores.values())))
    h_elo   = elos_snapshot[h]
    a_elo   = elos_snapshot[a]
    # Use only matches BEFORE this date to avoid look-ahead bias
    df_past = df_wc[df_wc['date'] < row['date']]
    h2w, h2d = h2h_stats(df_past, h, a)

    feature_rows.append({
        'strength_diff': round(h_score - a_score, 3),
        'elo_diff':      round(h_elo - a_elo, 1),
        'h2h_win_rate':  h2w,
        'h2h_draw_rate': h2d,
        'result':        row['result'],
    })
    # Update rolling Elo snapshot
    exp_h = expected_elo(elos_snapshot[h], elos_snapshot[a])
    act_h = 1.0 if row['result'] == 0 else (0.5 if row['result'] == 1 else 0.0)
    elos_snapshot[h] = update_elo(elos_snapshot[h], exp_h, act_h)
    elos_snapshot[a] = update_elo(elos_snapshot[a], 1-exp_h, 1-act_h)

df_feat = pd.DataFrame(feature_rows)
FEATURE_COLS = ['strength_diff', 'elo_diff', 'h2h_win_rate', 'h2h_draw_rate']

print(f"Feature matrix: {df_feat.shape[0]} rows × {len(FEATURE_COLS)} features")
print(f"Class distribution: {dict(df_feat['result'].value_counts().sort_index().rename({0:'Home W',1:'Draw',2:'Away W'}))}")
df_feat.describe().round(2)

---
## Part 3: Model Training

Two models trained on the historical World Cup feature matrix:

- **Logistic Regression** (`multinomial`, `lbfgs` solver) — interpretable, linear decision boundary. Feature importance via coefficients.
- **Random Forest** (500 trees, max_depth=10) — non-linear, captures interactions. Feature importance via mean impurity decrease.

Both use 5-fold cross-validation. Baseline accuracy for a 3-class random classifier = 33.3%.

In [ ]:
X = df_feat[FEATURE_COLS].values
y = df_feat['result'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)
X_all_sc = scaler.transform(X)   # for final CV on full dataset

print(f"Train: {len(X_train)} matches | Test: {len(X_test)} matches")
print(f"Features: {FEATURE_COLS}")

In [ ]:
lr = LogisticRegression(
    multi_class='multinomial', solver='lbfgs',
    max_iter=1000, C=1.0, random_state=42
)
lr.fit(X_tr_sc, y_train)

lr_cv   = cross_val_score(lr, X_all_sc, y, cv=5, scoring='accuracy')
lr_train = accuracy_score(y_train, lr.predict(X_tr_sc))
lr_test  = accuracy_score(y_test,  lr.predict(X_te_sc))

lr_coef_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'home_win':   lr.coef_[0].round(3),
    'draw':       lr.coef_[1].round(3),
    'away_win':   lr.coef_[2].round(3),
})

print("── Logistic Regression ─────────────────────────────────────────")
print(f"  CV accuracy  : {lr_cv.mean():.3f} ± {lr_cv.std():.3f}")
print(f"  Train acc    : {lr_train:.3f}")
print(f"  Test acc     : {lr_test:.3f}")
print(f"  Baseline (random 3-class): 0.333")
print("\nCoefficients (home-win class):")
print(lr_coef_df.to_string(index=False))

In [ ]:
rf = RandomForestClassifier(
    n_estimators=500, max_depth=10,
    min_samples_split=5, random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_cv    = cross_val_score(rf, X, y, cv=5, scoring='accuracy')
rf_train = accuracy_score(y_train, rf.predict(X_train))
rf_test  = accuracy_score(y_test,  rf.predict(X_test))

rf_imp_df = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': rf.feature_importances_.round(4),
}).sort_values('importance', ascending=False)

print("── Random Forest ───────────────────────────────────────────────")
print(f"  CV accuracy  : {rf_cv.mean():.3f} ± {rf_cv.std():.3f}")
print(f"  Train acc    : {rf_train:.3f}")
print(f"  Test acc     : {rf_test:.3f}")
print(f"  Baseline (random 3-class): 0.333")
print("\nFeature importances:")
print(rf_imp_df.to_string(index=False))

In [ ]:
comparison = pd.DataFrame({
    'Model':        ['Logistic Regression', 'Random Forest', 'Random baseline'],
    'CV Accuracy':  [lr_cv.mean(),           rf_cv.mean(),    1/3],
    'CV Std':       [lr_cv.std(),            rf_cv.std(),     0.0],
    'Train Acc':    [lr_train,               rf_train,        1/3],
    'Test Acc':     [lr_test,                rf_test,         1/3],
}).round(3)

print("── Model Comparison ────────────────────────────────────────────")
print(comparison.to_string(index=False))

best_model = 'Logistic Regression' if lr_cv.mean() >= rf_cv.mean() else 'Random Forest'
print(f"\n🏅 Best CV accuracy: {best_model}")

---
## Part 4: Tournament Simulation

The full 2026 bracket is simulated **twice** — once per model — then compared.

**2026 format:**
- 12 groups of 4 teams → 6 matches per group (72 total group stage matches)
- Top 2 from each group (24 teams) + 8 best third-place teams = **32 advance**
- Knockout rounds: Round of 32 → Round of 16 → Quarter-finals → Semi-finals → **Final**

In **group stage** matches, draws are possible (most likely outcome ≈ 25%).
In **knockout** matches, draw probability is redistributed proportionally between the two sides — same logic used in the 2022 project but now applied to ML output probabilities.

In [ ]:
def build_features(t1, t2, team_scores, team_elos):
    s1 = team_scores.get(t1, np.mean(list(team_scores.values())))
    s2 = team_scores.get(t2, np.mean(list(team_scores.values())))
    e1 = team_elos.get(t1, 1500.0)
    e2 = team_elos.get(t2, 1500.0)
    h2w, h2d = h2h_stats(df_wc, t1, t2)
    return np.array([[s1 - s2, e1 - e2, h2w, h2d]])

def simulate_match(t1, t2, team_scores, team_elos, is_knockout=False):
    feats = build_features(t1, t2, team_scores, team_elos)

    # LR probabilities
    feats_sc  = scaler.transform(feats)
    lr_probs  = lr.predict_proba(feats_sc)[0]
    # RF probabilities
    rf_probs  = rf.predict_proba(feats)[0]
    # class order: [0=home win, 1=draw, 2=away win]

    results = {}
    for name, probs in [('lr', lr_probs), ('rf', rf_probs)]:
        p_win, p_draw, p_lose = probs
        if is_knockout:
            total = p_win + p_lose
            adj_win  = p_win  + p_draw * (p_win  / total if total > 0 else 0.5)
            adj_lose = p_lose + p_draw * (p_lose / total if total > 0 else 0.5)
            winner = t1 if adj_win >= adj_lose else t2
            win_p  = max(adj_win, adj_lose)
            draw_p = 0.0
        else:
            idx = int(np.argmax(probs))
            winner = t1 if idx == 0 else (None if idx == 1 else t2)
            win_p  = probs[idx]
            draw_p = p_draw
        results[name] = {
            'winner': winner, 'win_prob': round(win_p, 3), 'draw_prob': round(draw_p, 3),
            'p_t1': round(p_win, 3), 'p_draw': round(p_draw, 3), 'p_t2': round(p_lose, 3),
        }
    return results   # {'lr': {...}, 'rf': {...}}

print("✅ simulate_match() ready")
print("   Returns dict with 'lr' and 'rf' keys, each containing winner + probabilities")

In [ ]:
def simulate_group(group_name, teams, team_scores, team_elos):
    tables = {m: {t: {'pts': 0, 'gd': 0, 'gf': 0} for t in teams} for m in ('lr', 'rf')}

    for t1, t2 in combinations(teams, 2):
        res = simulate_match(t1, t2, team_scores, team_elos, is_knockout=False)
        for m in ('lr', 'rf'):
            w = res[m]['winner']
            if w == t1:
                tables[m][t1]['pts'] += 3
                tables[m][t1]['gd'] += 1; tables[m][t1]['gf'] += 1
                tables[m][t2]['gd'] -= 1
            elif w == t2:
                tables[m][t2]['pts'] += 3
                tables[m][t2]['gd'] += 1; tables[m][t2]['gf'] += 1
                tables[m][t1]['gd'] -= 1
            else:  # draw
                tables[m][t1]['pts'] += 1
                tables[m][t2]['pts'] += 1

    return {
        m: sorted(tables[m].items(),
                  key=lambda x: (x[1]['pts'], x[1]['gd'], x[1]['gf']),
                  reverse=True)
        for m in ('lr', 'rf')
    }

# Simulate all 12 groups
group_standings = {}
for gname, teams in groups_2026.items():
    group_standings[gname] = simulate_group(gname, teams, team_scores, team_elos)

print("Group stage simulation complete. Standings (by points):")
for gname, stds in group_standings.items():
    lr_top = [t for t, _ in stds['lr'][:2]]
    print(f"  {gname}: LR top-2 → {', '.join(lr_top)}")

In [ ]:
def determine_advancement(group_standings, model='lr'):
    group_winners    = []
    group_runners_up = []
    third_place      = []

    for gname, stds in group_standings.items():
        s = stds[model]
        group_winners.append((s[0][0], gname, s[0][1]))
        group_runners_up.append((s[1][0], gname, s[1][1]))
        third_place.append((s[2][0], gname, s[2][1]))

    # 8 best third-place teams (by pts, then gd, then gf)
    best_thirds = sorted(
        third_place,
        key=lambda x: (x[2]['pts'], x[2]['gd'], x[2]['gf']),
        reverse=True
    )[:TOP_THIRDS]

    r32_teams = (
        group_winners +
        group_runners_up +
        best_thirds
    )
    teams_advancing = [t[0] for t in r32_teams]
    return teams_advancing, group_winners, group_runners_up, best_thirds

adv_lr, gw_lr, gr_lr, bt_lr = determine_advancement(group_standings, 'lr')
adv_rf, gw_rf, gr_rf, bt_rf = determine_advancement(group_standings, 'rf')

print(f"Teams advancing (LR): {len(adv_lr)}")
print(f"  Group winners: {', '.join(t for t,_,_ in gw_lr[:6])} ...")
print(f"  Best 3rd:      {', '.join(t for t,_,_ in bt_lr)}")

In [ ]:
def simulate_knockout_round(teams, team_scores, team_elos, model='lr', round_name=''):
    assert len(teams) % 2 == 0, "Must have even number of teams"
    winners = []
    match_results = []
    for i in range(0, len(teams), 2):
        t1, t2 = teams[i], teams[i+1]
        res = simulate_match(t1, t2, team_scores, team_elos, is_knockout=True)
        w   = res[model]['winner']
        winners.append(w)
        match_results.append({
            'round': round_name, 't1': t1, 't2': t2,
            'winner': w,
            'p_t1': res[model]['p_t1'],
            'p_draw': res[model]['p_draw'],
            'p_t2': res[model]['p_t2'],
        })
    return winners, match_results

def simulate_tournament(advancing_teams, team_scores, team_elos, model='lr'):
    bracket = {}
    all_matches = []
    current = list(advancing_teams)
    np.random.shuffle(current)   # random seeding within the bracket

    rounds = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Final']
    for rnd in rounds:
        if len(current) < 2:
            break
        winners, matches = simulate_knockout_round(current, team_scores, team_elos, model, rnd)
        bracket[rnd] = {'matches': matches, 'winners': winners}
        all_matches.extend(matches)
        current = winners

    champion = current[0] if current else 'Unknown'
    return champion, bracket, pd.DataFrame(all_matches)

# Run both models
champion_lr, bracket_lr, matches_lr = simulate_tournament(adv_lr, team_scores, team_elos, 'lr')
champion_rf, bracket_rf, matches_rf = simulate_tournament(adv_rf, team_scores, team_elos, 'rf')

print(f"🏆 LR predicts champion: {champion_lr}")
print(f"🏆 RF predicts champion: {champion_rf}")

In [ ]:
# Build agreement summary
all_lr = {r: [m['winner'] for m in info['matches']] for r, info in bracket_lr.items()}
all_rf = {r: [m['winner'] for m in info['matches']] for r, info in bracket_rf.items()}

agree_count, total_count = 0, 0
disagreements = []

for rnd in bracket_lr:
    w_lr = all_lr.get(rnd, [])
    w_rf = all_rf.get(rnd, [])
    matches_l = bracket_lr[rnd]['matches']
    matches_r = bracket_rf[rnd]['matches']
    for i, (ml, mr) in enumerate(zip(matches_l, matches_r)):
        total_count += 1
        agree = ml['winner'] == mr['winner']
        if agree:
            agree_count += 1
        else:
            disagreements.append({
                'round': rnd,
                'match': f"{ml['t1']} vs {ml['t2']}",
                'LR winner': ml['winner'], 'LR prob': ml['p_t1'],
                'RF winner': mr['winner'], 'RF prob': mr['p_t1'],
            })

print(f"\n{'═'*60}")
print(f"  2026 WORLD CUP PREDICTION RESULTS")
print(f"{'═'*60}")
print(f"  🏆 Logistic Regression: {champion_lr}")
print(f"  🏆 Random Forest:       {champion_rf}")
print(f"  📊 Agreement: {agree_count}/{total_count} knockout matches ({agree_count/total_count*100:.0f}%)")
print(f"{'─'*60}")

print(f"\nWhere models DISAGREE ({len(disagreements)} matches):")
for d in disagreements:
    print(f"  [{d['round']}] {d['match']}")
    print(f"    LR → {d['LR winner']} ({d['LR prob']:.0%})  |  RF → {d['RF winner']} ({d['RF prob']:.0%})")

print("\nKnockout bracket (LR):")
for rnd, info in bracket_lr.items():
    print(f"  {rnd}: {' | '.join(info['winners'])}")

---
## Part 5: Visualizations

Five charts — exported as PNG to `visualizations/`.

1. **Team Strength Rankings** — all 48 teams by composite score, coloured by confederation
2. **Model Accuracy** — LR vs RF cross-validation accuracy with error bars
3. **Feature Importance** — LR coefficients + RF importances side-by-side
4. **Bracket Agreement Map** — where the two models agree vs disagree
5. **Choropleth Map** — how far each team is predicted to go (one per model)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor(C['bg'])
ax.set_facecolor(C['bg'])

df_plot = df_team_scores.sort_values('score', ascending=True).copy()
df_plot['conf'] = df_plot['team'].map(TEAM_CONF).fillna('OTHER')
colors = [CONF_COLORS.get(c, C['muted']) for c in df_plot['conf']]

bars = ax.barh(df_plot['team'], df_plot['score'], color=colors, alpha=0.88, edgecolor='white', linewidth=0.4)

ax.set_xlabel('Composite Score (70/30 weighted)', fontsize=11, color=C['dark'])
ax.set_title('2026 World Cup — Team Strength Rankings', fontsize=15, fontweight='bold',
             color=C['dark'], pad=14)
ax.tick_params(axis='both', labelsize=8.5, colors=C['dark'])
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.spines['bottom'].set_color(C['border'] if 'border' in C else C['muted'])
ax.xaxis.set_minor_locator(plt.MultipleLocator(1))

legend_handles = [mpatches.Patch(color=v, label=k) for k, v in CONF_COLORS.items()]
ax.legend(handles=legend_handles, title='Confederation', fontsize=8, title_fontsize=9,
          loc='lower right', framealpha=0.7, facecolor=C['bg'])

plt.tight_layout()
out = VIZ_DIR / 'team_strength.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=C['bg'])
plt.show()
print(f"Saved: {out}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor(C['bg'])
ax.set_facecolor(C['bg'])

models  = ['Logistic\nRegression', 'Random\nForest']
means   = [lr_cv.mean(), rf_cv.mean()]
stds    = [lr_cv.std(),  rf_cv.std()]
colors2 = [C['accent'], C['teal']]

bars = ax.bar(models, means, yerr=stds, color=colors2, alpha=0.85,
              capsize=7, edgecolor='white', linewidth=0.8, width=0.45)

ax.axhline(1/3, color=C['dark'], linestyle='--', linewidth=1.2, alpha=0.6, label='Random baseline (33.3%)')

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.004,
            f'{mean:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold', color=C['dark'])

ax.set_ylabel('Cross-validation Accuracy (5-fold)', fontsize=10, color=C['dark'])
ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold', color=C['dark'], pad=10)
ax.set_ylim(0, max(means) + max(stds) + 0.08)
ax.legend(fontsize=9, framealpha=0.7, facecolor=C['bg'])
ax.tick_params(colors=C['dark'])
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
out = VIZ_DIR / 'model_accuracy.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=C['bg'])
plt.show()
print(f"Saved: {out}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.patch.set_facecolor(C['bg'])
for ax in (ax1, ax2):
    ax.set_facecolor(C['bg'])

# LR — coefficients for home-win class
lr_coefs = lr_coef_df.sort_values('home_win', ascending=True)
bar_colors = [C['accent'] if v >= 0 else C['cool'] for v in lr_coefs['home_win']]
ax1.barh(lr_coefs['feature'], lr_coefs['home_win'], color=bar_colors, alpha=0.85, edgecolor='white')
ax1.axvline(0, color=C['dark'], linewidth=0.8)
ax1.set_title('Logistic Regression\nCoefficients (home-win class)', fontsize=10, fontweight='bold', color=C['dark'])
ax1.tick_params(axis='both', labelsize=9, colors=C['dark'])
ax1.spines[['top','right']].set_visible(False)

# RF — feature importances
rf_imp = rf_imp_df.sort_values('importance', ascending=True)
ax2.barh(rf_imp['feature'], rf_imp['importance'], color=C['teal'], alpha=0.85, edgecolor='white')
ax2.set_title('Random Forest\nFeature Importances', fontsize=10, fontweight='bold', color=C['dark'])
ax2.tick_params(axis='both', labelsize=9, colors=C['dark'])
ax2.spines[['top','right']].set_visible(False)

fig.suptitle('What Drives Match Outcomes?', fontsize=13, fontweight='bold', color=C['dark'], y=1.02)
plt.tight_layout()
out = VIZ_DIR / 'feature_importance.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=C['bg'])
plt.show()
print(f"Saved: {out}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.patch.set_facecolor(C['bg'])
fig.suptitle('Predicted Bracket: Where Models Agree vs Disagree', fontsize=13,
             fontweight='bold', color=C['dark'], y=1.02)

AGREE_COLOR    = '#B8E0C0'  # soft green
DISAGREE_COLOR = '#F5C6B8'  # soft orange

round_order = ['Round of 32', 'Round of 16', 'Quarter-finals', 'Semi-finals', 'Final']
for col_idx, rnd in enumerate(round_order):
    ax = axes[col_idx]
    ax.set_facecolor(C['bg'])
    ax.set_title(rnd, fontsize=8.5, fontweight='bold', color=C['dark'])
    ax.axis('off')

    matches_l = bracket_lr.get(rnd, {}).get('matches', [])
    matches_r = bracket_rf.get(rnd, {}).get('matches', [])

    n = len(matches_l)
    if n == 0:
        continue

    for row_idx, (ml, mr) in enumerate(zip(matches_l, matches_r)):
        agree = ml['winner'] == mr['winner']
        bg    = AGREE_COLOR if agree else DISAGREE_COLOR
        y_pos = 1.0 - (row_idx / max(n, 1))

        # Match box
        rect = mpatches.FancyBboxPatch(
            (0.0, y_pos - 0.08), 1.0, 0.14,
            boxstyle="round,pad=0.01", linewidth=0.6,
            edgecolor=C['muted'], facecolor=bg, transform=ax.transAxes, clip_on=False
        )
        ax.add_patch(rect)

        lr_txt = f"LR: {ml['winner']} ({ml['p_t1']:.0%})"
        rf_txt = f"RF: {mr['winner']} ({mr['p_t1']:.0%})"
        matchup = f"{ml['t1']} vs\n{ml['t2']}"
        ax.text(0.5, y_pos + 0.02, matchup, ha='center', va='bottom',
                fontsize=5.5, color=C['dark'], transform=ax.transAxes, fontweight='bold')
        ax.text(0.5, y_pos - 0.03, lr_txt, ha='center', va='top',
                fontsize=4.5, color='#1A5C2A' if agree else '#8B2500', transform=ax.transAxes)
        ax.text(0.5, y_pos - 0.07, rf_txt, ha='center', va='top',
                fontsize=4.5, color='#1A5C2A' if agree else '#8B2500', transform=ax.transAxes)

legend_h = [
    mpatches.Patch(color=AGREE_COLOR,    label='Models agree'),
    mpatches.Patch(color=DISAGREE_COLOR, label='Models disagree'),
]
fig.legend(handles=legend_h, loc='lower center', ncol=2, fontsize=9,
           bbox_to_anchor=(0.5, -0.04), framealpha=0.8, facecolor=C['bg'])

plt.tight_layout()
out = VIZ_DIR / 'bracket_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor=C['bg'])
plt.show()
print(f"Saved: {out}")

In [ ]:
# Build progression depth (higher = further in tournament)
ROUND_SCORE = {'group_exit': 1, 'Round of 32': 2, 'Round of 16': 3,
               'Quarter-finals': 4, 'Semi-finals': 5, 'Final': 6, 'Winner': 7}

# Country name → ISO Alpha-3 (for Plotly choropleth)
COUNTRY_ISO = {
    'Argentina':'ARG','Brazil':'BRA','France':'FRA','England':'GBR','Germany':'DEU',
    'Spain':'ESP','Portugal':'PRT','Netherlands':'NLD','Belgium':'BEL','Croatia':'HRV',
    'Denmark':'DNK','Switzerland':'CHE','Uruguay':'URY','Colombia':'COL','Morocco':'MAR',
    'Senegal':'SEN','Japan':'JPN','South Korea':'KOR','United States':'USA','Mexico':'MEX',
    'Canada':'CAN','Ecuador':'ECU','Serbia':'SRB','Poland':'POL','Turkey':'TUR',
    'Austria':'AUT','Australia':'AUS','Iran':'IRN','Peru':'PER','Chile':'CHL',
    'Ivory Coast':'CIV','Nigeria':'NGA','Ghana':'GHA','Romania':'ROU','Cameroon':'CMR',
    'Czech Republic':'CZE','Tunisia':'TUN','Egypt':'EGY','Saudi Arabia':'SAU',
    'South Africa':'ZAF','Hungary':'HUN','Honduras':'HND','Panama':'PAN','Venezuela':'VEN',
    'Albania':'ALB','Jamaica':'JAM','New Zealand':'NZL','Ukraine':'UKR',
    'Iraq':'IRQ','Uzbekistan':'UZB','Scotland':'GBR','Jordan':'JOR','DR Congo':'COD',
}

def build_progression(bracket, all_teams, champion):
    prog = {t: 1 for t in all_teams}  # default: group exit
    for rnd, info in bracket.items():
        for m in info['matches']:
            for t in (m['t1'], m['t2']):
                prog[t] = max(prog.get(t, 1), ROUND_SCORE.get(rnd, 2) - 1)
            prog[m['winner']] = max(prog.get(m['winner'], 1), ROUND_SCORE.get(rnd, 2))
    if champion:
        prog[champion] = ROUND_SCORE['Winner']
    return prog

prog_lr = build_progression(bracket_lr, all_teams_2026, champion_lr)
prog_rf = build_progression(bracket_rf, all_teams_2026, champion_rf)

for label, prog, champ in [('LR', prog_lr, champion_lr), ('RF', prog_rf, champion_rf)]:
    df_map = pd.DataFrame([
        {'country': t, 'iso': COUNTRY_ISO.get(t, ''), 'depth': d,
         'stage': {v: k for k, v in ROUND_SCORE.items()}.get(d, 'Group exit')}
        for t, d in prog.items() if COUNTRY_ISO.get(t)
    ])

    fig_map = px.choropleth(
        df_map, locations='iso', color='depth',
        hover_name='country', hover_data={'stage': True, 'depth': False, 'iso': False},
        color_continuous_scale=[[0,'#F5E6C0'],[0.3,'#F49625'],[0.6,'#EF6545'],[1.0,'#422F0E']],
        title=f'2026 WC Predicted Progression — {label} Model (🏆 {champ})',
        range_color=[1, 7],
    )
    fig_map.update_layout(
        coloraxis_colorbar=dict(
            title='Round', tickvals=list(ROUND_SCORE.values()),
            ticktext=list(ROUND_SCORE.keys())
        ),
        geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'),
        font_family='sans-serif',
        paper_bgcolor=C['bg'],
    )
    out = VIZ_DIR / f'choropleth_{label.lower()}.png'
    if KALEIDO_AVAILABLE:
        fig_map.write_image(str(out), width=1200, height=600, scale=1.5)
        print(f"Saved: {out}")
    else:
        fig_map.show()
        print(f"(kaleido not installed — showing interactive map instead)")

---
## Part 6: Then vs Now — 2022 Deterministic vs 2026 ML

### 2022 Approach
- **Data**: Static FIFA 22 CSV (offline) + Wikipedia lineup scrape
- **Method**: Deterministic 70/30 weighted scoring — team with higher composite score always wins
- **Output**: One definite bracket. No uncertainty.
- **Result**: Correctly predicted France reaching the final
- **Limitation**: A team scoring 72.4 always beats a team scoring 72.3, regardless of how close that margin is. No concept of upsets, probability, or variance.

### 2026 Approach
- **Data**: Live API + historical WC match corpus (900+ matches) for ML training
- **Method**: Two competing probabilistic models (LR + Random Forest) trained on historical outcomes
- **Output**: Win probabilities per match — a 48% vs 52% matchup is treated very differently from 30% vs 70%
- **Upgrade**: Upsets are modelled. Confidence is quantified. The models disagree on some matches — that's the point.

### What Changed Technically

| | 2022 | 2026 |
|---|---|---|
| Data pipeline | Static CSV + Wikipedia | API + 900+ historical matches |
| Prediction logic | Rule-based comparison | Logistic Regression + Random Forest |
| Output | Definite bracket | Probability distributions |
| Uncertainty | None | Confidence scores per match |
| Training data | None (no learning) | 900+ historical WC matches |
| Teams / format | 32 teams, 8 groups | 48 teams, 12 groups, 8 best thirds |

### What Stayed the Same

The **core player-rating methodology** — the 70/30 weighted scoring algorithm — is **identical**.
Same `POSITION_WEIGHTS`, same `GENERAL_ATTRS`, same formula, just updated with FIFA 25 data.

This was deliberate: it **isolates the improvement**. The team-strength inputs are a constant;
what changed is the prediction engine built on top of them.

In [ ]:
# ── Print final results ───────────────────────────────────────────────────────
print("=" * 65)
print("  2026 FIFA WORLD CUP — FINAL PREDICTIONS")
print("=" * 65)
print(f"  🏆 Logistic Regression predicts: {champion_lr}")
print(f"  🏆 Random Forest predicts:       {champion_rf}")
print(f"  📊 Models agree on: {agree_count}/{total_count} knockout matches ({agree_count/total_count*100:.0f}%)")

# Key upsets (matches where favourite has < 60% win prob)
upsets_lr = matches_lr[matches_lr[['p_t1','p_t2']].max(axis=1) < 0.60]
if len(upsets_lr) > 0:
    print(f"\n  ⚠️  Close calls (LR, favourite <60%): {len(upsets_lr)} matches")
    for _, row in upsets_lr.head(5).iterrows():
        print(f"     [{row['round']}] {row['t1']} vs {row['t2']} → {row['winner']} ({max(row['p_t1'],row['p_t2']):.0%})")

print("=" * 65)

# ── Export to Excel ────────────────────────────────────────────────────────────
xlsx_path = DATA_DIR / 'wc2026_summary.xlsx'
with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
    df_team_scores.to_excel(writer, sheet_name='Team Scores', index=False)
    comparison.to_excel(writer, sheet_name='Model Comparison', index=False)
    matches_lr.to_excel(writer, sheet_name='LR Bracket', index=False)
    matches_rf.to_excel(writer, sheet_name='RF Bracket', index=False)
    pd.DataFrame(disagreements).to_excel(writer, sheet_name='Disagreements', index=False)
    pd.DataFrame([
        {'metric': 'LR Champion', 'value': champion_lr},
        {'metric': 'RF Champion', 'value': champion_rf},
        {'metric': 'LR CV Accuracy', 'value': f'{lr_cv.mean():.3f}'},
        {'metric': 'RF CV Accuracy', 'value': f'{rf_cv.mean():.3f}'},
        {'metric': 'Matches agreed', 'value': f'{agree_count}/{total_count}'},
        {'metric': 'Ratings source', 'value': RATINGS_SOURCE},
        {'metric': 'Generated', 'value': datetime.now().strftime('%Y-%m-%d %H:%M')},
    ]).to_excel(writer, sheet_name='Summary', index=False)

print(f"\n📊 Summary exported → {xlsx_path}")
print("\nVisualisations saved:")
for f in sorted(VIZ_DIR.iterdir()):
    print(f"  {f}")